# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {metadata.cite_as}")
print(f"Publication Date: {metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We first list the available record sets and then review the fields (columns) for each record set. All references are through their `@id` fields.

In [ ]:
# List all RecordSets, their @id, and their fields
def print_record_sets(dataset):
    for rs in dataset.record_sets:
        print(f"RecordSet name: {rs.name} @id: {rs.id}")
        print("Fields (columns):")
        for field in rs.fields:
            name = getattr(field, 'name', None)
            _id = getattr(field, 'id', None)
            dtype = getattr(field, 'data_type', None)
            print(f"    - {name} (@id: {getattr(field, 'id', None)}, type: {dtype})")
        print()

print_record_sets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We will load each record set and print the available columns. Please refer to the previous list of `@id`s for selecting further fields of interest.

In [ ]:
# Extract data from each record set
# Identify record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for a record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like outlier removal, transformations, and grouping.

For this demo, let's:
- Choose the first record set found above and a numeric field (such as abundance or MW) by its `@id`.
- Filter records where the numeric value is greater than a threshold.
- Normalize the numeric field.
- Group the data by another categorical field (e.g., protein name/description if present).

In [ ]:
# ---------- CUSTOMIZE THESE IDs BASED ON THE DATA OVERVIEW ABOVE ---------- #
# Set the target record set @id, a numeric field @id, and a group field @id
if len(record_set_ids) == 0:
    raise ValueError("No record sets found in the dataset.")
record_set_id = record_set_ids[0]  # Use the first record set

df = dataframes[record_set_id]
print("First few columns in this record set:", df.columns.tolist())

# Attempt to find a numeric field (heuristic)
numeric_field = None
for col in df.columns:
    s = pd.to_numeric(df[col], errors='coerce')
    if s.notnull().all() or s.notnull().sum() > len(df)*0.5:  # Mostly numbers
        numeric_field = col
        break
if numeric_field is None:
    raise ValueError("No numeric field found for demonstration.")

group_field = None
# Try to find a non-numeric, categorical field
for col in df.columns:
    if col != numeric_field and df[col].dtype == object:
        group_field = col
        break
if group_field is None:
    group_field = df.columns[0]  # fallback

print(f"Chosen numeric field for EDA: {numeric_field}")
print(f"Grouping field: {group_field}")

# Filter: only keep rows with numeric_field > threshold
threshold = pd.to_numeric(df[numeric_field], errors='coerce').mean()
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
display(filtered_df.head())

# Normalize
filtered_numeric = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
filtered_df[f"{numeric_field}_normalized"] = (filtered_numeric - filtered_numeric.mean()) / filtered_numeric.std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by the group_field and show statistics
if group_field in df.columns:
    grouped_df = (filtered_df.groupby(group_field)
                 [numeric_field, f"{numeric_field}_normalized"].mean().reset_index())
    print(f"Grouped data by {group_field} (mean values):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the filtered and normalized numeric field, and a group-wise aggregation if relevant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution (after filtering)
plt.figure(figsize=(8,4))
sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field has low cardinality, plot mean per group
if 'grouped_df' in locals() and grouped_df[group_field].nunique() <= 20:
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant-described dataset with `mlcroissant`, inspect metadata, explore available record sets and fields via their `@id`, extract and analyze data in `pandas`, and visualize basic data distributions.

**Key findings:**
- The dataset comprises mass spectrometry-based protein data with fields such as sequence descriptors, molecular weights, and abundance.
- Data can be filtered and analyzed interactively using only schema-defined `@id` references, ensuring reproducibility and clarity.

**Next steps:**
- Apply domain-specific filtering and visualizations
- Perform downstream ML or statistical tasks using curated data
- Map complete Croissant provenance chain for full reproducibility

Explore further with additional fields and custom analyses using the field `@id` lookup approach outlined above!